### Generating Ground Truth Data

To evaluate search, we need a dataset of queries where we know which document is the correct answer. This is called ground truth (or gold standard) data.

For each query in our ground truth dataset, we know which document in the knowledge base is relevant. When we run a search, we check whether the results include the correct document.

There are several ways to get ground truth data:

* Human annotators look at documents and write queries (best quality, expensive)
* Collect real user queries and label them (requires a running system)
* Generate synthetic data with an LLM (what we'll do)


We don't have a production system yet, so we'll use an LLM to generate questions. For each FAQ document, we ask the LLM to create 5 questions that this document would answer. Then we know that for each generated question, the source document is the correct answer.


#### Loading the documents

We'll use helper files.

Then load the FAQ data:

In [1]:
from ingest import load_faq_data
documents = load_faq_data()

We'll generate questions only for the LLM Zoomcamp FAQ. The full FAQ dataset contains documents from multiple courses. 

Generating five questions for every document would take longer and cost more.

In [2]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

103

We'll use these documents from now on so let's name them as documents

In [3]:
documents = documents_llm

In [5]:
documents

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.",
  'doc_id': '977bf7786c'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
  'answer': 'The zoom lin

Each document already has an id field:

In [6]:
doc = documents[0]
print(doc["doc_id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


The ID becomes the label in our ground truth dataset. We generate questions from a document, so we know that this document holds the answer. 

Later, search evaluation checks whether search brings back the document with this ID.

This is why every record needs a stable ID. If you can't uniquely identify a document, you can't tell whether search retrieved the right one. 


When you build your own evaluation set, *assign an ID to each record* in your knowledge base first.

#### Generating questions with structured output

We use an LLM to generate questions for each document.

This is the first time we're using structured output in the course. With structured output, we ask the LLM to return data in a specific format instead of free-form text. 

For example, instead of getting a paragraph that contains questions, we can ask for a Python object with a questions field.

This is useful when code will process the output. The model returns the same structure every time. We can access the generated questions directly instead of parsing text manually.

We want the output as a list of strings, so we define that structure with a Pydantic model:

In [7]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

The instructions for the LLM:

In [8]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

We ask the LLM to use different wording from the original document. This makes the evaluation more realistic - real users won't phrase their questions the same way as the FAQ.

Call the LLM for one document:

In [10]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

Prepare the document as JSON:

In [11]:
import json

user_prompt = json.dumps(doc)

Create the messages:

In [13]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

Until now we called responses.create and read response.output_text. 

For structured output we switch to responses.parse and pass text_format=Questions, which tells the API to return our class instead of free text.

Call the model:

In [14]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

The parsed object is available in response.output_parsed:

In [15]:
result = response.output_parsed

print(result)

questions=['Can I join the course if I just found out about it, or is it too late?', 'Is it still okay to start the course now even though it already began?', 'If I join late, can I still get a certificate somehow?', 'Do I need to submit my project before submissions close to qualify for the certificate?', 'What happens if I start the course now but miss the project submission deadline?']


We can access the list directly:

In [16]:
print(result.questions)

['Can I join the course if I just found out about it, or is it too late?', 'Is it still okay to start the course now even though it already began?', 'If I join late, can I still get a certificate somehow?', 'Do I need to submit my project before submissions close to qualify for the certificate?', 'What happens if I start the course now but miss the project submission deadline?']


You should see 5 questions that relate to the first FAQ document.

#### Reusable utilities

We'll need this pattern again in other evaluation sections today, so we put it in a reusable helper.

It contains helper functions we'll reuse in this module:

* llm_structured: calls the OpenAI API with structured output
* llm_structured_retry: retries structured-output calls when a request fails
* calc_price: calculates the price from token usage
* calc_total_price: calculates the total price from multiple usage objects
* map_progress: runs work in parallel and tracks progress. We'll use it in the next lesson.

Import the structured-output helper:

In [17]:
from evaluation_utils import llm_structured

Use it on the same document:

In [18]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course — can I still join it now?', 'If I join late, is there still a way to get a certificate?', 'Do I need to submit my project before submissions close to earn the certificate?', 'Is it okay to start the course after it has already begun?', 'What do I need to do if I want the certificate after joining late?']


#### Tracking cost

The response also contains token usage:

In [19]:
usage.input_tokens, usage.output_tokens

(207, 87)

As in the agents module, we calculate the price from response.usage.

Import the price helper:

In [20]:
from evaluation_utils import calc_price

Calculate the cost of this call:

In [21]:
cost = calc_price(usage)

cost

{'input_cost': 0.00015525,
 'output_cost': 0.00039150000000000003,
 'total_cost': 0.00054675}

Now convert these questions into ground truth records:

In [23]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["doc_id"]
    })

records

[{'question': 'I just found this course — can I still join it now?',
  'document': '74eb249bbf'},
 {'question': 'If I join late, is there still a way to get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to submit my project before submissions close to earn the certificate?',
  'document': '74eb249bbf'},
 {'question': 'Is it okay to start the course after it has already begun?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do if I want the certificate after joining late?',
  'document': '74eb249bbf'}]

Each record has two fields:

* question: the question generated by the LLM
* document: the ID of the FAQ document that should answer the question

The document field connects the generated question to the document that contains the answer. 

Later, when we evaluate search, we'll ask the search engine the generated question. 

Then we'll check if it retrieves the document with this ID.

We now know how to generate and store questions for one document. In the next lesson, we'll run this for all LLM Zoomcamp FAQ documents and save the full ground truth dataset.

### Generating Ground Truth for All Documents

In the previous part, we generated questions for one document and converted them into ground truth records.

We want to do the same thing for every document in the FAQ dataset. For each document, we generate questions and save them as ground truth records.


For this part, we'll use tqdm for progress bars and pandas for saving the final CSV.


If you don't have them installed yet, add them first:

uv add tqdm pandas

The processing function takes one document and turns it into ground truth records.

For each document, we:

* convert the document to JSON so we can send it to the LLM
* ask the LLM to return a Questions object
* create one ground truth record for each generated question

Each record contains the generated question and the ID of the document that should answer the question.

When we send many requests, one of them might fail. We don't want the entire batch to fail because of one temporary error.

Import the retry helper from evaluation_utils.py:

In [24]:
from evaluation_utils import llm_structured_retry

llm_structured makes one structured-output call. 

llm_structured_retry wraps the same call in a retry loop. 

If one request fails because of a temporary API or network issue, it waits briefly and tries again.

Use it in the processing function:

In [27]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["doc_id"]
        })

    return results, usage

Try it for the first 5 documents.

Import tqdm and run the loop:

In [28]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

This works, but it runs one LLM call after another. Running it for all documents this way would take too long.